# 13 - EF-Primary R(2+1)D Baseline

This notebook trains an EF-only R(2+1)D baseline on deterministic full cardiac-cycle clips from EchoNet-Dynamic. It is intentionally independent of segmentation, ConvLSTM, LSTM, and motion supervision. The existing ConvLSTM segmentation checkpoint is loaded only as a frozen evaluation/localization resource for future XAI work.


## Cardiac-Cycle Window Definition

For each video, ED and ES frame indices are read from `VolumeTracings.csv`. Because the standard EchoNet table stores traced frame numbers but not always explicit `EDFrame`/`ESFrame` columns, ED is assigned to the traced frame with the larger LV polygon area and ES to the traced frame with the smaller LV polygon area.

The window is video-specific and now prefers a true-cycle-style span. If ED occurs before ES, the preferred window is `ED -> estimated next ED`, where estimated next ED is `ED + 2 * abs(ED - ES)`. If that would exceed the video boundary, the code falls back to an estimated previous-cycle window ending at ED. The selected window is uniformly resampled to 32 frames, and the nearest sampled positions are snapped to ED and ES so the clip retains both annotated phases whenever possible. Original sampled frame indices and normalized temporal positions are saved for inference and XAI.



## Setup


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import json
import os
import random
import sys
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def first_existing_path(candidates):
    cleaned = [candidate for candidate in candidates if candidate]
    for candidate in cleaned:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(cleaned[-1])


PROJECT_ROOT = first_existing_path([
    os.environ.get("PROJECT_ROOT"),
    "/kaggle/input/echonet-temporal-xai",
    "/kaggle/input/src-updated",
    "/kaggle/working/Echonet_temporal_XAI",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bidirectional_convlstm_unet import BidirectionalConvLSTMUNet
from src.cardiac_cycle_dataset import (
    CardiacCycleWindowConfig,
    EchoNetCardiacCycleDataset,
    build_cardiac_cycle_manifest,
)
from src.r2plus1d_ef import (
    R2Plus1DEFRegressor,
    denormalize_ef,
    evaluate_ef,
    make_grad_scaler,
    reversed_clip,
    shuffled_clip,
    spatially_blurred_clip,
    temporally_subsampled_repeat_clip,
    train_one_epoch_ef,
    zero_motion_clip,
)
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
VIDEOS_DIR = RAW_DIR / "Videos"
SEGMENTATION_CHECKPOINT_DIR = Path(os.environ.get(
    "BIDIRECTIONAL_CONVLSTM_CHECKPOINT_DIR",
    PROJECT_ROOT / "outputs" / "runs" / "bidirectional_convlstm_unet_23_frames" / "checkpoints",
))
RUN_DIR = Path(os.environ.get(
    "RUN_DIR",
    "/kaggle/working/outputs/runs/r2plus1d_ef_baseline" if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "r2plus1d_ef_baseline",
))
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MANIFEST_DIR = RUN_DIR / "manifests"
FIGURES_DIR = RUN_DIR / "figures"
FEATURE_DIR = RUN_DIR / "features"
for directory in [RUN_DIR, CHECKPOINT_DIR, MANIFEST_DIR, FIGURES_DIR, FEATURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Segmentation checkpoint directory: {SEGMENTATION_CHECKPOINT_DIR}")
print(f"Output directory: {RUN_DIR}")



## Configuration


In [ ]:
RUN_MODE = "smoke"  # change to "full" for complete training
CLIP_LENGTH = 32
IMAGE_SIZE = (112, 112)

SMOKE_CONFIG = {
    "run_mode": "smoke",
    "seed": 42,
    "clip_length": CLIP_LENGTH,
    "image_size": list(IMAGE_SIZE),
    "cycle_scale": 2.0,
    "min_window_frames": 32,
    "force_include_ed_es": True,
    "prefer_true_cycle": True,
    "max_videos": 32,
    "max_train_videos": 16,
    "max_val_videos": 8,
    "max_test_videos": 8,
    "batch_size": 2,
    "num_workers": 2,
    "epochs": 1,
    "lr": 1e-4,
    "weight_decay": 1e-5,
    "dropout": 0.2,
    "hidden_dim": 256,
    "pretrained_backbone": True,
    "mixed_precision": True,
    "save_test_features": False,
    "feature_layers": ["layer3", "layer4", "final_spatiotemporal"],
    "perturbation_seed": 42,
}

FULL_CONFIG = {
    **SMOKE_CONFIG,
    "run_mode": "full",
    "max_videos": None,
    "max_train_videos": None,
    "max_val_videos": None,
    "max_test_videos": None,
    "batch_size": 4,
    "epochs": 30,
    "lr": 3e-5,
    "save_test_features": False,
}

config = SMOKE_CONFIG if RUN_MODE == "smoke" else FULL_CONFIG
set_seed(config["seed"])
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)
config




## Load EchoNet Tables and Build Cardiac-Cycle Manifest


In [ ]:
assert (RAW_DIR / "FileList.csv").exists(), f"Missing FileList.csv under {RAW_DIR}"
assert (RAW_DIR / "VolumeTracings.csv").exists(), f"Missing VolumeTracings.csv under {RAW_DIR}"
assert VIDEOS_DIR.exists(), f"Missing Videos directory: {VIDEOS_DIR}"
file_list, volume_tracings = load_echonet_tables(RAW_DIR)
window_config = CardiacCycleWindowConfig(
    clip_length=config["clip_length"],
    image_size=tuple(config["image_size"]),
    cycle_scale=config["cycle_scale"],
    min_window_frames=config["min_window_frames"],
    force_include_ed_es=config["force_include_ed_es"],
    prefer_true_cycle=config["prefer_true_cycle"],
)
manifest_df = build_cardiac_cycle_manifest(
    file_list=file_list,
    volume_tracings=volume_tracings,
    videos_dir=VIDEOS_DIR,
    config=window_config,
    max_videos=config["max_videos"],
)
assert not manifest_df.empty, "No videos available after building cardiac-cycle manifest."
manifest_df.to_csv(MANIFEST_DIR / "cardiac_cycle_manifest.csv", index=False)
coverage_summary = {
    "video_count": int(len(manifest_df)),
    "clip_length": int(config["clip_length"]),
    "window_rule": "prefer ED-to-estimated-next-ED true-cycle window using length=max(2*ED_ES_distance, min_window_frames, clip_length); if unavailable near video boundary, use estimated previous-cycle window ending at ED; clamp to video bounds; uniformly resample; snap nearest sampled positions to ED and ES",
    "percentage_window_contains_ed": float(100.0 * manifest_df["window_contains_ed"].mean()),
    "percentage_window_contains_es": float(100.0 * manifest_df["window_contains_es"].mean()),
    "percentage_window_contains_both_ed_es": float(100.0 * manifest_df["window_contains_both_ed_es"].mean()),
    "percentage_sampled_contains_ed": float(100.0 * manifest_df["contains_ed_exact_sampled"].mean()),
    "percentage_sampled_contains_es": float(100.0 * manifest_df["contains_es_exact_sampled"].mean()),
    "percentage_sampled_contains_both_ed_es": float(100.0 * manifest_df["contains_both_ed_es_exact_sampled"].mean()),
    "ed_es_distance_min": float(manifest_df["ed_es_distance_frames"].min()),
    "ed_es_distance_median": float(manifest_df["ed_es_distance_frames"].median()),
    "ed_es_distance_mean": float(manifest_df["ed_es_distance_frames"].mean()),
    "ed_es_distance_max": float(manifest_df["ed_es_distance_frames"].max()),
    "window_length_min": float(manifest_df["window_length_frames"].min()),
    "window_length_median": float(manifest_df["window_length_frames"].median()),
    "window_length_mean": float(manifest_df["window_length_frames"].mean()),
    "window_length_max": float(manifest_df["window_length_frames"].max()),
}
with (MANIFEST_DIR / "cardiac_cycle_coverage_summary.json").open("w", encoding="utf-8") as file:
    json.dump(coverage_summary, file, indent=2)
print(json.dumps(coverage_summary, indent=2))
display(manifest_df[["video_id", "split", "ef", "ed_frame_idx", "es_frame_idx", "window_start_frame", "window_end_frame", "sampled_frame_indices"]].head())



## Split Manifest and DataLoaders


In [ ]:
train_manifest = manifest_df[manifest_df["split"] == "TRAIN"].reset_index(drop=True)
val_manifest = manifest_df[manifest_df["split"].isin(["VAL", "VALIDATION"])].reset_index(drop=True)
test_manifest = manifest_df[manifest_df["split"] == "TEST"].reset_index(drop=True)
assert min(len(train_manifest), len(val_manifest), len(test_manifest)) > 0, {
    "train": len(train_manifest), "validation": len(val_manifest), "test": len(test_manifest)
}
if config["max_train_videos"] is not None:
    train_manifest = train_manifest.iloc[:config["max_train_videos"]].reset_index(drop=True)
if config["max_val_videos"] is not None:
    val_manifest = val_manifest.iloc[:config["max_val_videos"]].reset_index(drop=True)
if config["max_test_videos"] is not None:
    test_manifest = test_manifest.iloc[:config["max_test_videos"]].reset_index(drop=True)

train_manifest.to_csv(MANIFEST_DIR / "train_manifest.csv", index=False)
val_manifest.to_csv(MANIFEST_DIR / "validation_manifest.csv", index=False)
test_manifest.to_csv(MANIFEST_DIR / "test_manifest.csv", index=False)
ef_mean = float(train_manifest["ef"].mean())
ef_std = float(train_manifest["ef"].std(ddof=0))
assert ef_std > 0, "Training EF standard deviation is zero."
print(f"Active train/val/test videos: {len(train_manifest)} / {len(val_manifest)} / {len(test_manifest)}")
print(f"EF normalization: mean={ef_mean:.3f}, std={ef_std:.3f}")

train_dataset = EchoNetCardiacCycleDataset(train_manifest, image_size=tuple(config["image_size"]), ef_mean=ef_mean, ef_std=ef_std)
val_dataset = EchoNetCardiacCycleDataset(val_manifest, image_size=tuple(config["image_size"]), ef_mean=ef_mean, ef_std=ef_std)
test_dataset = EchoNetCardiacCycleDataset(test_manifest, image_size=tuple(config["image_size"]), ef_mean=ef_mean, ef_std=ef_std)
loader_kwargs = {
    "batch_size": config["batch_size"],
    "num_workers": config["num_workers"],
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": config["num_workers"] > 0,
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
sample = train_dataset[0]
assert sample["video"].shape == (3, config["clip_length"], *tuple(config["image_size"]))
assert sample["sampled_frame_indices"].shape[0] == config["clip_length"]
assert sample["sampled_normalized_positions"].shape[0] == config["clip_length"]
print({k: tuple(v.shape) for k, v in sample.items() if torch.is_tensor(v) and v.ndim > 0})


## Frozen ConvLSTM Segmentation Model for Future LV Localization


In [ ]:
def select_checkpoint(checkpoint_dir: Path) -> Path:
    for name in ["best_model.pt", "best_val_dice_model.pt", "final_model.pt"]:
        path = checkpoint_dir / name
        if path.exists():
            return path
    candidates = sorted(path for path in checkpoint_dir.iterdir() if path.suffix in {".pt", ".pth", ".ckpt"})
    if not candidates:
        raise FileNotFoundError(f"No checkpoint found in {checkpoint_dir}")
    return candidates[0]

segmentation_model = BidirectionalConvLSTMUNet(
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    num_frames_before=11,
    num_frames_after=11,
).to(device)
seg_ckpt_path = select_checkpoint(SEGMENTATION_CHECKPOINT_DIR)
seg_ckpt = torch.load(seg_ckpt_path, map_location="cpu")
seg_state = seg_ckpt.get("model_state_dict", seg_ckpt.get("state_dict", seg_ckpt))
if any(key.startswith("module.") for key in seg_state):
    seg_state = {key.removeprefix("module."): value for key, value in seg_state.items()}
load_result = segmentation_model.load_state_dict(seg_state, strict=False)
assert not load_result.missing_keys, load_result.missing_keys
assert not load_result.unexpected_keys, load_result.unexpected_keys
for param in segmentation_model.parameters():
    param.requires_grad = False
segmentation_model.eval()
seg_report = {"checkpoint_path": str(seg_ckpt_path), "trainable_parameters": int(sum(p.numel() for p in segmentation_model.parameters() if p.requires_grad))}
with (MANIFEST_DIR / "frozen_segmentation_model_report.json").open("w", encoding="utf-8") as file:
    json.dump(seg_report, file, indent=2)
print(json.dumps(seg_report, indent=2))


## Build R(2+1)D EF Model


In [ ]:
model = R2Plus1DEFRegressor(
    pretrained=config["pretrained_backbone"],
    dropout=config["dropout"],
    hidden_dim=config["hidden_dim"],
).to(device)
loss_fn = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
scaler = make_grad_scaler(enabled=bool(config["mixed_precision"] and torch.cuda.is_available()))
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Stable feature layers:", model.feature_layer_names, "plus final_spatiotemporal and pooled_features")


## Validate Forward, Feature Return, and Gradient Flow


In [ ]:
validation_batch = next(iter(train_loader))
video = validation_batch["video"].to(device)[:1].clone().detach().requires_grad_(True)
out = model(video, return_features=True, feature_layers=tuple(config["feature_layers"]))
assert out["ef"].shape == (1,)
assert "pooled_features" in out
assert "features" in out and "final_spatiotemporal" in out["features"]
final_feature = out["features"]["final_spatiotemporal"]
final_feature.retain_grad()
out["ef"].sum().backward()
assert video.grad is not None and torch.isfinite(video.grad).all() and float(video.grad.abs().sum()) > 0
assert final_feature.grad is not None and torch.isfinite(final_feature.grad).all() and float(final_feature.grad.abs().sum()) > 0
print({
    "input_video": tuple(video.shape),
    "ef": tuple(out["ef"].shape),
    "pooled_features": tuple(out["pooled_features"].shape),
    "final_spatiotemporal": tuple(final_feature.shape),
    "input_grad_l1": float(video.grad.abs().sum().detach().cpu()),
    "feature_grad_l1": float(final_feature.grad.abs().sum().detach().cpu()),
})
model.zero_grad(set_to_none=True)


## Train EF-Only Baseline


In [ ]:
history = []
best_val_mae = float("inf")
for epoch in range(1, config["epochs"] + 1):
    train_metrics = train_one_epoch_ef(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=device,
        ef_mean=ef_mean,
        ef_std=ef_std,
        mixed_precision=config["mixed_precision"],
        scaler=scaler,
    )
    val_metrics, _ = evaluate_ef(model, val_loader, loss_fn, device, ef_mean, ef_std)
    row = {"epoch": epoch, **train_metrics, **{k.replace("eval_", "val_"): v for k, v in val_metrics.items()}}
    history.append(row)
    history_df = pd.DataFrame(history)
    history_df.to_csv(RUN_DIR / "history.csv", index=False)
    payload = {"epoch": epoch, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "metrics": row, "config": config, "ef_mean": ef_mean, "ef_std": ef_std}
    torch.save(payload, CHECKPOINT_DIR / "final_model.pt")
    if row["val_ef_mae"] < best_val_mae:
        best_val_mae = row["val_ef_mae"]
        torch.save(payload, CHECKPOINT_DIR / "best_ef_mae_model.pt")
    print(f"Epoch {epoch:03d}/{config['epochs']:03d} train_loss={row['train_loss']:.4f} val_EF_MAE={row['val_ef_mae']:.2f} val_EF_RMSE={row['val_ef_rmse']:.2f} val_r={row['val_ef_pearson']:.3f}")
display(pd.DataFrame(history).tail())


## Evaluate Normal Test Set


In [ ]:
best_ckpt = torch.load(CHECKPOINT_DIR / "best_ef_mae_model.pt", map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()
test_metrics, test_predictions = evaluate_ef(model, test_loader, loss_fn, device, ef_mean, ef_std)
test_metrics = {k.replace("eval_", "test_"): v for k, v in test_metrics.items()}
test_predictions_df = pd.DataFrame(test_predictions)
test_predictions_df.to_csv(MANIFEST_DIR / "test_predictions.csv", index=False)
with (RUN_DIR / "test_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(test_metrics, file, indent=2)
print(json.dumps(test_metrics, indent=2))
display(test_predictions_df.head())


## Temporal Perturbation Evaluation


In [ ]:
def zero_motion_from_labelled_ed(video: torch.Tensor, batch: dict[str, Any]) -> torch.Tensor:
    out = torch.empty_like(video)
    sampled = batch["sampled_frame_indices"]
    ed_frames = batch["ed_frame_idx"]
    for i in range(video.shape[0]):
        matches = torch.where(sampled[i] == ed_frames[i])[0]
        source_index = int(matches[0]) if len(matches) else 0
        out[i:i + 1] = zero_motion_clip(video[i:i + 1], source_index=source_index)
    return out

PERTURBATION_CONDITIONS = [
    {"name": "normal_cardiac_cycle_clip", "transform": None},
    {"name": "zero_motion_labelled_ed_frame", "transform": zero_motion_from_labelled_ed},
    {"name": "randomly_shuffled_frame_order", "transform": partial(shuffled_clip, seed=config["perturbation_seed"])},
    {"name": "reversed_temporal_order", "transform": reversed_clip},
    {"name": "temporally_subsampled_repeat", "transform": partial(temporally_subsampled_repeat_clip, step=2)},
    {"name": "spatially_blurred_clip", "transform": spatially_blurred_clip},
]
condition_rows = []
condition_prediction_tables = []
for condition in PERTURBATION_CONDITIONS:
    metrics, predictions = evaluate_ef(
        model=model,
        loader=test_loader,
        loss_fn=loss_fn,
        device=device,
        ef_mean=ef_mean,
        ef_std=ef_std,
        sequence_transform=condition["transform"],
    )
    row = {"condition": condition["name"], **{k.replace("eval_", ""): v for k, v in metrics.items()}}
    condition_rows.append(row)
    pred_df = pd.DataFrame(predictions)
    pred_df["condition"] = condition["name"]
    condition_prediction_tables.append(pred_df)
condition_df = pd.DataFrame(condition_rows)
normal = condition_df[condition_df["condition"] == "normal_cardiac_cycle_clip"].iloc[0]
for metric in ["ef_mae", "ef_rmse"]:
    condition_df[f"normal_{metric}"] = float(normal[metric])
    condition_df[f"delta_{metric}_vs_normal"] = condition_df[metric] - float(normal[metric])
condition_df.to_csv(MANIFEST_DIR / "temporal_perturbation_metrics.csv", index=False)
condition_predictions_df = pd.concat(condition_prediction_tables, ignore_index=True)
condition_predictions_df.to_csv(MANIFEST_DIR / "temporal_perturbation_predictions.csv", index=False)
display(condition_df)



## Optional Feature Export for XAI


In [ ]:
if config["save_test_features"]:
    model.eval()
    rows = []
    for batch in tqdm(test_loader, desc="save features"):
        video = batch["video"].to(device, non_blocking=True)
        with torch.no_grad():
            out = model(video, return_features=True, feature_layers=tuple(config["feature_layers"]))
        ef_pred = denormalize_ef(out["ef"].detach().cpu(), ef_mean, ef_std).numpy()
        for i, video_id in enumerate(batch["video_id"]):
            out_path = FEATURE_DIR / f"{video_id}_features.npz"
            arrays = {
                "ef_pred": np.array(float(ef_pred[i]), dtype=np.float32),
                "sampled_frame_indices": batch["sampled_frame_indices"][i].numpy().astype(np.int32),
                "sampled_normalized_positions": batch["sampled_normalized_positions"][i].numpy().astype(np.float32),
                "pooled_features": out["pooled_features"][i].detach().cpu().numpy().astype(np.float32),
            }
            for name, tensor in out["features"].items():
                arrays[f"feature_{name}"] = tensor[i].detach().cpu().numpy().astype(np.float32)
            np.savez_compressed(out_path, **arrays)
            rows.append({"video_id": str(video_id), "feature_path": str(out_path.relative_to(RUN_DIR))})
    feature_manifest_df = pd.DataFrame(rows)
else:
    feature_manifest_df = pd.DataFrame(columns=["video_id", "feature_path"])
feature_manifest_df.to_csv(MANIFEST_DIR / "feature_manifest.csv", index=False)
display(feature_manifest_df.head())


## Plots


In [ ]:
history_df = pd.read_csv(RUN_DIR / "history.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].set_title("Smooth L1 loss")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["val_ef_mae"], label="val EF MAE")
axes[1].plot(history_df["epoch"], history_df["val_ef_rmse"], label="val EF RMSE")
axes[1].set_title("Validation EF metrics")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(test_predictions_df["ef_true"], test_predictions_df["ef_pred"], s=14, alpha=0.7)
lo = min(test_predictions_df["ef_true"].min(), test_predictions_df["ef_pred"].min())
hi = max(test_predictions_df["ef_true"].max(), test_predictions_df["ef_pred"].max())
ax.plot([lo, hi], [lo, hi], color="black", linewidth=1)
ax.set_xlabel("Ground-truth EF (%)")
ax.set_ylabel("Predicted EF (%)")
ax.set_title("R(2+1)D EF prediction")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "test_ef_pred_vs_true.png", dpi=150, bbox_inches="tight")
plt.close(fig)


## Required Outputs


In [ ]:
required_outputs = [
    RUN_DIR / "config.json",
    RUN_DIR / "history.csv",
    RUN_DIR / "test_metrics.json",
    CHECKPOINT_DIR / "best_ef_mae_model.pt",
    CHECKPOINT_DIR / "final_model.pt",
    MANIFEST_DIR / "cardiac_cycle_manifest.csv",
    MANIFEST_DIR / "cardiac_cycle_coverage_summary.json",
    MANIFEST_DIR / "train_manifest.csv",
    MANIFEST_DIR / "validation_manifest.csv",
    MANIFEST_DIR / "test_manifest.csv",
    MANIFEST_DIR / "test_predictions.csv",
    MANIFEST_DIR / "temporal_perturbation_metrics.csv",
    MANIFEST_DIR / "temporal_perturbation_predictions.csv",
    MANIFEST_DIR / "feature_manifest.csv",
    MANIFEST_DIR / "frozen_segmentation_model_report.json",
    FIGURES_DIR / "training_curves.png",
    FIGURES_DIR / "test_ef_pred_vs_true.png",
]
missing = [str(path) for path in required_outputs if not path.exists()]
assert not missing, f"Missing expected outputs: {missing}"
print(f"All outputs saved under: {RUN_DIR}")
